# 深度学习中的主要 Norm

本笔记本从四个角度说明神经网络中使用的**主要 norm / 归一化方法**：

1. **数学**：norm 是什么以及如何计算  
2. **原理**：为何有助于优化或稳定性  
3. **放置位置**：放在**功能层之前**还是**之后**  
4. **失效模式**：若不使用会怎样，以及会出现哪些训练病态  

此外还包括：

- 具体数值示例
- 清晰的结构示意图
- 按网络类型整理的总结
- 将症状映射到可能归一化问题的调试表

---

## 重要区分

“norm” 有两个相关但不同的含义：

### A. 数学范数（Mathematical norms）
它们度量向量或矩阵的**大小**，例如：

- $L_1$
- $L_2$
- $L_\infty$
- Frobenius norm
- Spectral norm

### B. 神经网络中的归一化层 / 机制
它们在训练过程中主动控制尺度与统计量，例如：

- BatchNorm
- LayerNorm
- RMSNorm
- GroupNorm
- InstanceNorm
- WeightNorm
- SpectralNorm

前者是**数学对象**。  
后者是**训练 / 架构层面的机制**。

# 1. 数学范数（Mathematical Norms）

范数（norm）是度量大小的函数 $\|\cdot\|$。

对于向量 $x$，合法的范数必须满足：

1. **非负性（Non-negativity）**
   $$
   \|x\| \ge 0,\qquad \|x\|=0 \iff x=0
   $$

2. **齐次性（Homogeneity）**
   $$
   \|\alpha x\| = |\alpha|\,\|x\|
   $$

3. **三角不等式（Triangle inequality）**
   $$
   \|x+y\| \le \|x\|+\|y\|
   $$

这些条件使该量具有类似广义长度的行为。

## 1.1 向量范数

设

$$
x = \begin{bmatrix} 3 \\ -4 \end{bmatrix}
$$

### $L_1$ norm
$$
\|x\|_1 = \sum_i |x_i|
$$

因此，

$$
\|x\|_1 = |3| + |-4| = 7
$$

**Interpretation：** 各分量绝对值之和，即总绝对幅度

---

### $L_2$ norm
$$
\|x\|_2 = \sqrt{\sum_i x_i^2}
$$

因此，

$$
\|x\|_2 = \sqrt{3^2 + (-4)^2} = 5
$$

**Interpretation：** Euclidean length（欧氏长度）

---

### $L_\infty$ norm
$$
\|x\|_\infty = \max_i |x_i|
$$

因此，

$$
\|x\|_\infty = 4
$$

**Interpretation：** 最大分量的大小

## 1.2 矩阵范数

设

$$
A =
\begin{bmatrix}
1 & 2 \\
3 & 4
\end{bmatrix}
$$

### Frobenius norm
$$
\|A\|_F = \sqrt{\sum_{i,j} A_{ij}^2}
$$

因此，

$$
\|A\|_F = \sqrt{1^2 + 2^2 + 3^2 + 4^2} = \sqrt{30}
$$

**Interpretation：** 将矩阵展平后对其取 $L_2$ norm

---

### Spectral norm
$$
\|A\|_2 = \sigma_{\max}(A)
$$

其中 $\sigma_{\max}(A)$ 为最大奇异值。

等价定义：

$$
\|A\|_2 = \max_{x\ne 0} \frac{\|Ax\|_2}{\|x\|_2}
$$

**Interpretation：** 线性映射的最大放大因子

若 $\|A\|_2$ 很大，则存在某个方向使得该层强烈放大信号。

In [1]:
import numpy as np

x = np.array([3, -4], dtype=float)
A = np.array([[1, 2], [3, 4]], dtype=float)

l1 = np.sum(np.abs(x))
l2 = np.linalg.norm(x, 2)
linf = np.linalg.norm(x, np.inf)
fro = np.linalg.norm(A, 'fro')
spec = np.linalg.norm(A, 2)

print("x =", x)
print("||x||_1 =", l1)
print("||x||_2 =", l2)
print("||x||_inf =", linf)
print()
print("A =")
print(A)
print("||A||_F =", fro)
print("spectral norm ||A||_2 =", spec)

x = [ 3. -4.]
||x||_1 = 7.0
||x||_2 = 5.0
||x||_inf = 4.0

A =
[[1. 2.]
 [3. 4.]]
||A||_F = 5.477225575051661
spectral norm ||A||_2 = 5.464985704219043


# 2. 为何深度网络需要 Normalization

深度网络反复施加变换：

$$
x_{\ell+1} = f_\ell(x_\ell)
$$

在反向传播中，梯度流经 Jacobian 的乘积：

$$
\frac{\partial \mathcal{L}}{\partial x_0}
=
\frac{\partial \mathcal{L}}{\partial x_L}
\prod_{\ell=1}^{L} J_\ell
$$

若每层典型的放大倍数略大于 $1$，信号可能爆炸。

若略小于 $1$，信号可能消失。

例如：

- 若每层约放大 $1.3$ 倍，则经过 $20$ 层后：
  $$
  1.3^{20} \approx 190
  $$

- 若每层约缩小到 $0.7$ 倍，则经过 $20$ 层后：
  $$
  0.7^{20} \approx 0.0008
  $$

这就是归一化方法如此重要的基本原因：

- 稳定激活的尺度
- 稳定梯度流
- 降低对初始化的敏感度
- 常允许使用更大的学习率
- 改善优化的条件数（optimization conditioning）

# 3. 清晰结构示意图

## 3.1 CNN 风格块

典型模式：

```text
Input
  ↓
Conv / Linear
  ↓
Norm   (BN / GN / IN, sometimes LN)
  ↓
Activation   (ReLU / GELU / SiLU)
  ↓
Next layer
```

归一化通常放在：

> **在线性/卷积层之后、激活之前**

## 3.2 标准 CNN 残差块

```text
x
├─────────────── skip ───────────────┐
│                                    ↓
└→ Conv → BN → ReLU → Conv → BN → Add → ReLU
```

这是经典的 ResNet 风格模式。

## 3.3 Transformer Post-Norm 结构

```text
x
│
├→ Attention / MLP
│
└────────────── Add residual ──────────────→ LayerNorm
```

公式：

$$
y = \mathrm{LN}(x + \mathrm{Sublayer}(x))
$$

归一化的位置：

> **在功能子层之后，且在残差相加之后**

## 3.4 Transformer Pre-Norm 结构

```text
x
│
├→ LayerNorm / RMSNorm → Attention / MLP
│
└────────────── Add residual ──────────────→ y
```

公式：

$$
y = x + \mathrm{Sublayer}(\mathrm{LN}(x))
$$

或

$$
y = x + \mathrm{Sublayer}(\mathrm{RMSNorm}(x))
$$

归一化的位置：

> **在功能子层之前**

这是现代深层 Transformer 和 LLM 的主流设计。

---

### Pre-Norm 与 Post-Norm：区别

| 属性 | Post-Norm | Pre-Norm |
|---|---|---|
| **LN 位置** | 在残差相加之后 | 在子层之前 |
| **公式** | $y = \mathrm{LN}(x + \mathrm{Sublayer}(x))$ | $y = x + \mathrm{Sublayer}(\mathrm{LN}(x))$ |
| **训练稳定性** | 较不稳定，需要 warmup | 更稳定，常可省去 warmup |
| **LR 敏感度** | 高，需仔细调参 | 低，对 LR 选择更鲁棒 |
| **梯度流** | 每层梯度都须经过 LN | 残差路径清晰，梯度直接传递 |
| **最终性能** | 调参得当时略优 | 略差，但随规模差距缩小 |
| **深度扩展** | 难以训练极深模型 | 可轻松扩展到数百层 |
| **代表模型** | 原始 Transformer、BERT | GPT-2/3、LLaMA、PaLM、多数现代 LLM |

**为何 Pre-Norm 更稳定：**

在 Pre-Norm 中，残差连接在输出与输入之间形成直接的相加路径：

$$
y = x + \mathrm{Sublayer}(\mathrm{LN}(x))
$$

反向传播时，$\partial y / \partial x$ 中始终包含恒等项，因此梯度可沿残差流传递，而无需在每一层都被 LayerNorm 重新缩放。在 Post-Norm 中，每个梯度都必须经过 LN，多层重复的缩放可能导致梯度不稳定。

**为何 Post-Norm 有时能略获更好的最终性能：**

残差相加之后的 LayerNorm 会对每层输出分布起到正则化作用，有助于各层表示更加一致。Pre-Norm 则使最后一级子层输出未经归一化，可能对表示空间的利用略不充分。不过这一差距较小，且随模型规模增大而缩小，因此几乎所有现代 LLM 仍因稳定性优势而选择 Pre-Norm。


## 3.5 WeightNorm / SpectralNorm

二者并非作为单独的激活归一化层插入。

它们作用于**层自身的权重**。

```text
Input → Linear/Conv(with normalized/reparameterized weights) → Output
```

因此对这两种方法而言，问「之前还是之后？」并不恰当。

更恰当的理解是：

> **权重级别的归一化/重参数化**


# 4. BatchNorm (BN)

## 4.1 数学

对于一个 mini-batch 中某个通道的激活值 $x_1,\dots,x_m$：

$$
\mu_B = \frac{1}{m}\sum_{i=1}^{m} x_i
$$

$$
\sigma_B^2 = \frac{1}{m}\sum_{i=1}^{m}(x_i-\mu_B)^2
$$

归一化：

$$
\hat{x}_i = \frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}
$$

然后对每个通道 $c$ 施加可学习的仿射参数：

$$
y_{i,c} = \gamma_c \, \hat{x}_{i,c} + \beta_c
$$

此处 $\gamma \in \mathbb{R}^C$ 与 $\beta \in \mathbb{R}^C$ 为向量，每个通道对应一个元素，因此可学习参数总数为 $2C$（而不仅仅是两个标量）。

---

## 4.2 数值示例

考虑一个包含 $N=2$ 个样本的 mini-batch，每个样本有 $C=2$ 个特征通道，空间尺寸为 $1 \times 1$（即经全局平均池化之后，或简单视为全连接层输出）。

激活张量的形状为 $(N, C) = (2, 2)$：

$$
X = \begin{bmatrix} 1 & 10 \\ 3 & 20 \end{bmatrix}
$$

其中第 $i$ 行为样本 $i$，第 $c$ 列为通道 $c$。

**BatchNorm 对每个通道在整个 batch 上独立归一化。**

### 通道 0：值为 $[1, 3]$

$$
\mu_0 = \frac{1+3}{2} = 2, \qquad
\sigma_0^2 = \frac{(1-2)^2+(3-2)^2}{2} = 1
$$

$$
\hat{x}_{:,0} = \frac{[1,3]-2}{\sqrt{1}} = [-1,\; 1]
$$

### 通道 1：值为 $[10, 20]$

$$
\mu_1 = \frac{10+20}{2} = 15, \qquad
\sigma_1^2 = \frac{(10-15)^2+(20-15)^2}{2} = 25
$$

$$
\hat{x}_{:,1} = \frac{[10,20]-15}{\sqrt{25}} = [-1,\; 1]
$$

### 施加逐通道可学习参数

设 $\gamma = [\gamma_0, \gamma_1] = [2, 3]$ 且 $\beta = [\beta_0, \beta_1] = [0.5, -1]$。

每个通道使用各自的 $\gamma_c, \beta_c$：

$$
y_{:,0} = 2 \cdot [-1, 1] + 0.5 = [-1.5,\; 2.5]
$$

$$
y_{:,1} = 3 \cdot [-1, 1] + (-1) = [-4,\; 2]
$$

这说明通道 0 和通道 1 使用**不同的统计量**进行归一化（$\mu_0 \ne \mu_1$，$\sigma_0 \ne \sigma_1$），然后通过**不同的可学习参数**进行缩放和偏移（$\gamma_0 \ne \gamma_1$，$\beta_0 \ne \beta_1$）。


In [2]:
X = np.array([[1., 10.],
              [3., 20.]])  # shape (N=2, C=2)

gamma = np.array([2., 3.])    # per-channel scale
beta  = np.array([0.5, -1.])  # per-channel shift

print("BatchNorm example: N=2 samples, C=2 channels")
print("X (shape N x C):")
print(X)
print()

for c in range(X.shape[1]):
    col = X[:, c]
    mu  = col.mean()
    var = ((col - mu) ** 2).mean()
    std = np.sqrt(var)
    xhat = (col - mu) / std
    y = gamma[c] * xhat + beta[c]
    print(f"--- Channel {c} ---")
    print(f"  values  = {col}")
    print(f"  mean    = {mu}")
    print(f"  var     = {var}")
    print(f"  std     = {std}")
    print(f"  normed  = {xhat}")
    print(f"  gamma_{c} = {gamma[c]},  beta_{c} = {beta[c]}")
    print(f"  output  = {y}")
    print()

BatchNorm example: N=2 samples, C=2 channels
X (shape N x C):
[[ 1. 10.]
 [ 3. 20.]]

--- Channel 0 ---
  values  = [1. 3.]
  mean    = 2.0
  var     = 1.0
  std     = 1.0
  normed  = [-1.  1.]
  gamma_0 = 2.0,  beta_0 = 0.5
  output  = [-1.5  2.5]

--- Channel 1 ---
  values  = [10. 20.]
  mean    = 15.0
  var     = 25.0
  std     = 5.0
  normed  = [-1.  1.]
  gamma_1 = 3.0,  beta_1 = -1.0
  output  = [-4.  2.]



## 4.3 BN 的放置位置

典型 CNN 模式：

```text
Conv → BN → ReLU
```

因此 BN 放在：

> **在卷积/全连接层之后、激活函数之前**

为什么？

因为 BN 在非线性之前稳定了激活值的分布。

---

## 4.4 BN 解决的问题

BN 主要有助于：

- 各层之间激活尺度不稳定
- 对学习率敏感
- 深层 CNN 收敛缓慢
- 优化过程中条件数不佳（poor conditioning）

---

## 4.5 使用 BN 的典型网络

- CNN
- ResNet
- 图像分类 backbone
- 许多经典视觉架构

---

## 4.6 没有 BN 时的典型病症

深层 CNN 中常见的训练病态：

- 激活值爆炸或漂移
- 梯度不稳定
- 损失振荡
- 对初始化极度敏感
- 可用学习率显著变小
- 收敛变慢或无法收敛


# 5. LayerNorm (LN)

## 5.1 Mathematics

LayerNorm 在**单个样本内部**、沿特征维度进行归一化。

对于隐藏向量 $x = [x_1,\dots,x_d]$：

$$
\mu = \frac{1}{d}\sum_{j=1}^{d} x_j
$$

$$
\sigma^2 = \frac{1}{d}\sum_{j=1}^{d} (x_j-\mu)^2
$$

则

$$
\hat{x}_j = \frac{x_j-\mu}{\sqrt{\sigma^2+\epsilon}}
$$

并且

$$
y_j = \gamma_j \hat{x}_j + \beta_j
$$

---

## 5.2 Numerical example

设

$$
x = [1,2,3]
$$

则

$$
\mu = 2
$$

$$
\sigma^2 = \frac{(1-2)^2+(2-2)^2+(3-2)^2}{3} = \frac{2}{3}
$$

$$
\sqrt{\sigma^2} \approx 0.816
$$

因此

$$
\hat{x} \approx [-1.225,0,1.225]
$$


In [3]:
x = np.array([1, 2, 3], dtype=float)
mu = x.mean()
var = ((x - mu) ** 2).mean()
std = np.sqrt(var)
xhat = (x - mu) / std

print("LayerNorm example")
print("x =", x)
print("mean =", mu)
print("var =", var)
print("std =", std)
print("normalized =", xhat)

LayerNorm example
x = [1. 2. 3.]
mean = 2.0
var = 0.6666666666666666
std = 0.816496580927726
normalized = [-1.22474487  0.          1.22474487]


## 5.3 LN 的放置位置

### 在普通 MLP 块中
常见形式为：

```text
Linear → LN → GELU
```

因此此处 LN 的放置方式为：

> **在线性层之后、激活函数之前**

### 在 Transformer 中
有两种主要布局。

#### Post-Norm
```text
x → Sublayer → Add residual → LN
```

#### Pre-Norm
```text
x → LN → Sublayer → Add residual
```

现代大型 Transformer 通常采用 **Pre-Norm**，因为它对深层网络更稳定。

---

## 5.4 LN 解决的问题

- 隐藏状态尺度漂移
- 残差流幅度不稳定
- 当 batch 统计不可靠时训练效果差
- 序列模型与 Transformer 中的不稳定

---

## 5.5 使用 LN 的典型网络

- Transformers
- ViTs
- 序列模型
- 基于注意力机制的网络

---

## 5.6 没有 LN 时的典型病症

常见病症：

- 残差流幅度漂移
- attention logits 不稳定
- softmax 饱和
- 深层扩展更难
- 对 warmup、初始化与学习率更敏感


## 5.7 BatchNorm vs LayerNorm：它们在哪个维度上归一化？

根本区别在于**计算统计量的轴不同**。

对于形状为 $(N, d)$ 的张量——$N$ 个样本，$d$ 个特征：

```text
         特征维度 →
       ┌────┬────┬────┬────┐
  n=0  │    │    │    │    │ ── LayerNorm：沿这一行归一化
       ├────┼────┼────┼────┤
  n=1  │    │    │    │    │ ── LayerNorm：沿这一行归一化
       ├────┼────┼────┼────┤
  n=2  │    │    │    │    │ ── LayerNorm：沿这一行归一化
       └────┴────┴────┴────┘
         │    │    │    │
         BN   BN   BN   BN   ← BatchNorm：沿每一列归一化
```

- **BatchNorm**：对每个特征通道 $c$，在所有 $N$ 个样本上计算 $\mu_c, \sigma_c^2$ → **逐通道，跨 batch**
- **LayerNorm**：对每个样本 $n$，在所有 $d$ 个特征上计算 $\mu_n, \sigma_n^2$ → **逐样本，跨特征**

---

### 基于相同数据的数值对比

使用 $X = \begin{bmatrix} 1 & 10 \\ 3 & 20 \end{bmatrix}$，形状为 $(N\!=\!2,\; d\!=\!2)$：

| | BatchNorm（按列） | LayerNorm（按行） |
|---|---|---|
| **切片** | 列 0：$[1, 3]$，列 1：$[10, 20]$ | 行 0：$[1, 10]$，行 1：$[3, 20]$ |
| **$\mu$** | $\mu_0=2,\; \mu_1=15$ | $\mu_0=5.5,\; \mu_1=11.5$ |
| **$\sigma^2$** | $\sigma_0^2=1,\; \sigma_1^2=25$ | $\sigma_0^2=20.25,\; \sigma_1^2=72.25$ |
| **效果** | 同一特征 → 在不同样本间可比 | 同一样本 → 在不同特征间可比 |

---

### 关键行为差异

| 属性 | BatchNorm | LayerNorm |
|---|---|---|
| **是否依赖 batch** | 是——统计量来自当前 mini-batch | 否——每个样本独立 |
| **可学习参数** | $\gamma, \beta \in \mathbb{R}^C$（逐通道） | $\gamma, \beta \in \mathbb{R}^d$（逐特征维） |
| **训练 / 推理差异** | 有——训练用 batch 统计，推理用滑动统计 | 无——行为一致 |
| **小 batch** | 统计量嘈杂，质量下降 | 不受影响 |
| **batch size = 1** | 无意义（单样本的「batch 均值」） | 完全可用 |
| **变长序列** | padding 会污染 batch 统计 | 无此问题，按 token 计算 |
| **主要领域** | CNN（ResNet 等） | Transformers、RNNs、MLPs |

---

### 为什么 Transformer 使用 LayerNorm 而非 BatchNorm

1. **变长序列**：NLP batch 中句子长度不一且含 padding。BatchNorm 在计算列统计时会将真实 token 与 padding 混在一起。LayerNorm 在每个 token 内独立计算。

2. **推理一致性**：BatchNorm 存在训练/推理差异（batch 统计 vs 滑动统计）。LayerNorm 在两种模式下行为相同。

3. **自回归生成**：LLM 逐 token 生成，batch size 常为 1。BatchNorm 会退化为近似无操作。LayerNorm 仍可对 $d$ 维隐藏向量归一化。

---

### 什么时候 BatchNorm 更合适

若输入特征具有**不同的物理量纲或尺度差异很大**（例如弧度角 $\sim\mathcal{O}(1)$、米制距离 $\sim\mathcal{O}(10^3)$、角速度 $\sim\mathcal{O}(10^{-1})$），则逐通道归一化（BatchNorm 或手动逐特征标准化）更合适。

LayerNorm 将所有特征混合为单一的 $\mu, \sigma^2$。若某一特征尺度远大于其他特征，它会主导统计量，较小特征几乎不受影响——下游线性层的 Jacobian 谱仍接近秩 1。

**经验法则：**

> **异构特征尺度 → 逐通道（BatchNorm / 手动标准化）**
>
> **齐次隐藏表示 → 逐样本（LayerNorm / RMSNorm）**


In [4]:
X = np.array([[1., 10.],
              [3., 20.]])  # shape (N=2, d=2)

print("X (shape N x d):")
print(X)
print()

print("=" * 55)
print("BatchNorm: normalize each COLUMN (per channel, across batch)")
print("=" * 55)
for c in range(X.shape[1]):
    col = X[:, c]
    mu  = col.mean()
    var = ((col - mu) ** 2).mean()
    xhat = (col - mu) / np.sqrt(var)
    print(f"  channel {c}: values={col}  →  μ={mu}, σ²={var}  →  normed={xhat}")

print()
print("=" * 55)
print("LayerNorm: normalize each ROW (per sample, across features)")
print("=" * 55)
for n in range(X.shape[0]):
    row = X[n, :]
    mu  = row.mean()
    var = ((row - mu) ** 2).mean()
    xhat = (row - mu) / np.sqrt(var)
    print(f"  sample {n}: values={row}  →  μ={mu}, σ²={var}  →  normed={np.round(xhat, 4)}")

X (shape N x d):
[[ 1. 10.]
 [ 3. 20.]]

BatchNorm: normalize each COLUMN (per channel, across batch)
  channel 0: values=[1. 3.]  →  μ=2.0, σ²=1.0  →  normed=[-1.  1.]
  channel 1: values=[10. 20.]  →  μ=15.0, σ²=25.0  →  normed=[-1.  1.]

LayerNorm: normalize each ROW (per sample, across features)
  sample 0: values=[ 1. 10.]  →  μ=5.5, σ²=20.25  →  normed=[-1.  1.]
  sample 1: values=[ 3. 20.]  →  μ=11.5, σ²=72.25  →  normed=[-1.  1.]


# 6. RMSNorm

## 6.1 数学

RMSNorm 通过均方根进行归一化，不减去均值。

$$
\mathrm{RMS}(x) = \sqrt{\frac{1}{d}\sum_{j=1}^{d} x_j^2 + \epsilon}
$$

则

$$
y_j = \gamma_j \frac{x_j}{\mathrm{RMS}(x)}
$$

因此与 LayerNorm 不同，它**不会**显式地将激活值中心化。

---

## 6.2 数值示例

设

$$
x = [1,2,3]
$$

则

$$
\mathrm{RMS}(x) = \sqrt{\frac{1^2+2^2+3^2}{3}}
= \sqrt{\frac{14}{3}} \approx 2.160
$$

因此

$$
\frac{x}{\mathrm{RMS}(x)} \approx [0.463,0.926,1.389]
$$


In [5]:
x = np.array([1, 2, 3], dtype=float)
rms = np.sqrt(np.mean(x**2))
y = x / rms

print("RMSNorm example")
print("x =", x)
print("RMS =", rms)
print("normalized =", y)

RMSNorm example
x = [1. 2. 3.]
RMS = 2.160246899469287
normalized = [0.46291005 0.9258201  1.38873015]


## 6.3 RMSNorm 的放置位置

在现代 LLM 中，RMSNorm 通常按 **Pre-Norm** 模式放置：

```text
x → RMSNorm → Attention → Add residual
x → RMSNorm → MLP       → Add residual
```

因此 RMSNorm 放在：

> **功能子层之前**

---

## 6.4 RMSNorm 解决的问题

- 控制隐状态的幅度
- 稳定深层残差 Transformer
- 提供比 LN 更简洁的归一化
- 在大语言模型中效果很好

---

### 为什么 RMSNorm 不去中心化效果反而更好？

RMSNorm 相比 LayerNorm 省略了减去均值（centering）这一步，但在大模型中表现相当甚至更好。原因有三个层面：

### (1) 实证发现：去中心化对性能贡献极小

RMSNorm 原始论文（Zhang & Sennrich, 2019）将 LayerNorm 拆解为两步做消融实验，发现**去均值对最终性能的贡献微乎其微**，真正起作用的是缩放（除以某种幅度量）。在多个翻译和语言建模任务上，去掉均值减法后性能几乎没有下降。

### (2) 去中心化是冗余的——下游 bias 可以补偿

Transformer 中每个 Norm 后面紧跟线性变换 $Wx + b$。即使 Norm 没有减去均值，线性层的 bias $b$ 完全可以学到等价的偏移：

$$
W \cdot \frac{x - \mu}{\sigma} + b \quad \equiv \quad W \cdot \frac{x}{\mathrm{RMS}(x)} + b'
$$

网络可以通过调整 $b'$ 来补偿缺失的中心化。换言之，去中心化提供的信息对于有 bias 的线性层来说是冗余的。

此外，隐藏表示的均值本身可能携带有用信息（比如 token 的整体能量水平）。LayerNorm 强行减去均值相当于**丢弃了一个维度的信号**，而 RMSNorm 保留了它。

### (3) 计算效率

| 操作 | LayerNorm | RMSNorm |
|---|---|---|
| 均值 $\mu = \frac{1}{d}\sum x_j$ | 需要 | 不需要 |
| 减均值 $x - \mu$ | 需要 | 不需要 |
| 方差 $\sigma^2 = \frac{1}{d}\sum(x_j - \mu)^2$ | 需要（两次遍历） | 不需要 |
| RMS $= \sqrt{\frac{1}{d}\sum x_j^2}$ | 不需要 | 需要（一次遍历） |
| 可学习 bias $\beta$ | 有 | 通常省略 |
| **总计 reduce 操作** | **2 次**（mean + var） | **1 次**（mean of squares） |

对于 $d = 4096$（LLaMA-7B）甚至 $d = 12288$（LLaMA-65B），每个 token 每层省掉一次 reduce + 一次逐元素减法，累积节省显著。

### 核心直觉

归一化层的核心目标不是让分布变成 $\mathcal{N}(0,1)$，而是**控制幅度（magnitude）**，防止残差流的 scale 在深层累积后爆炸或消失。

- LayerNorm 做了两件事：(1) 中心化 (2) 缩放
- RMSNorm 只做一件事：缩放

而"控制幅度"只需要缩放就够了。中心化的好处被下游线性层的 bias 吸收，成本却是实打实的。因此 RMSNorm 是一个**净收益的 trade-off**——省计算、不损性能。

这也是为什么 LLaMA、PaLM 等现代 LLM 全部采用 RMSNorm 而非 LayerNorm。

---

## 6.5 使用 RMSNorm 的典型网络

- LLMs
- decoder-only Transformer
- LLaMA 风格架构

---

## 6.6 深层 LLM 中没有 RMSNorm/LN 时的典型病症

- 残差流尺度会增长或漂移
- attention / MLP 子层数值不稳定
- 训练更脆弱
- 深层扩展更困难


# 7. GroupNorm (GN)

## 7.1 数学

GroupNorm 将通道分成若干组，在每组内部进行归一化。

若有 $C$ 个通道和 $G$ 个组，则每组包含 $C/G$ 个通道。

在每个样本和每组内计算：

$$
\hat{x} = \frac{x-\mu_g}{\sqrt{\sigma_g^2+\epsilon}}
$$

然后施加可学习的仿射参数。

---

## 7.2 GN 的放置位置

典型模式：

```text
Conv → GN → ReLU
```

因此 GN 放在：

> **在卷积之后、激活函数之前**

---

## 7.3 GN 解决的问题

GN 在 batch size 较小、BN 统计量噪声较大时特别有用。

有助于：

- 小 batch CNN 训练中稳定归一化
- 检测/分割场景
- 避免依赖 batch 统计量

---

## 7.4 使用 GN 的典型网络

- 目标检测
- 分割
- 小 batch size 的 CNN

---

## 7.5 小 batch CNN 中没有 GN 时的典型病症

如果在很小的 batch 上使用 BN，常见问题包括：

- 统计量噪声大
- loss 不稳定
- 精度下降
- train/eval 不一致问题更严重


# 8. InstanceNorm (IN)

## 8.1 数学

InstanceNorm 对每个样本的每个通道在空间维度上分别进行归一化。

对于样本 $n$ 和通道 $c$：

$$
\hat{x}_{n,c,:,:}
=
\frac{x_{n,c,:,:}-\mu_{n,c}}{\sqrt{\sigma_{n,c}^2+\epsilon}}
$$

---

## 8.2 IN 的放置位置

典型模式：

```text
Conv → IN → Activation
```

因此 IN 放在：

> **在卷积之后、激活函数之前**

---

## 8.3 IN 解决的问题

当需要移除或控制风格统计量时，InstanceNorm 很有用，尤其在图像生成/风格迁移中。

有助于：

- 减少实例特有的风格变化
- 聚焦于内容结构
- 提高风格迁移的一致性

---

## 8.4 使用 IN 的典型网络

- 风格迁移
- 图到图生成
- 艺术图像合成

---

## 8.5 风格相关任务中没有 IN 时的典型病症

- 源图像风格泄漏
- 风格化弱化
- 纹理/颜色迁移不一致

---

## 8.6 GroupNorm、LayerNorm、InstanceNorm 的关系

这三者其实是同一种 **per-sample 归一化** 的不同特例，区别仅在于"将多少个通道分为一组"：

```text
InstanceNorm ←────── GroupNorm ──────→ LayerNorm
  (G = C)           (1 < G < C)          (G = 1)
```

| 方法 | 组数 $G$ | 每组通道数 | 归一化范围 |
|---|---|---|---|
| **InstanceNorm** | $G = C$ | 1 个通道 | 每个样本、每个通道的空间维度 |
| **GroupNorm** | $1 < G < C$ | $C/G$ 个通道 | 每个样本、每组内的通道 + 空间 |
| **LayerNorm** | $G = 1$ | 全部 $C$ 个通道 | 每个样本的所有通道 + 空间 |

**核心共性：三者都在单个样本内部计算统计量，完全不依赖 batch。**

用图来看（shape $(N, C)$，$C=4$，$G=2$）：

```text
         C=0   C=1   C=2   C=3
       ┌─────┬─────┬─────┬─────┐
  n=0  │  a  │  b  │  c  │  d  │
       ├─────┼─────┼─────┼─────┤
  n=1  │  e  │  f  │  g  │  h  │
       └─────┴─────┴─────┴─────┘

InstanceNorm (G=4): 每个格子独立 → {a}, {b}, {c}, {d}, ...
GroupNorm    (G=2): 每两列一组   → {a,b}, {c,d}, {e,f}, {g,h}
LayerNorm    (G=1): 整行一组     → {a,b,c,d}, {e,f,g,h}
```

### 与 BatchNorm 的根本区别

BatchNorm 是**纵向**操作（跨样本），GroupNorm/LN/IN 是**横向**操作（样本内部）。方向完全不同：

```text
         C=0   C=1   C=2   C=3
       ┌─────┬─────┬─────┬─────┐
  n=0  │     │     │     │     │ ── GN/LN/IN：在这一行内归一化
       ├─────┼─────┼─────┼─────┤
  n=1  │     │     │     │     │ ── GN/LN/IN：在这一行内归一化
       └─────┴─────┴─────┴─────┘
         │     │     │     │
         BN    BN    BN    BN    ← BatchNorm：沿每一列跨样本归一化
```

因此 **GroupNorm 是 LayerNorm 和 InstanceNorm 之间的泛化**，而非 BatchNorm 的扩展。它解决的是 BatchNorm 在小 batch 下统计量噪声过大导致性能崩溃的问题，通过完全不依赖 batch 来绕开这一限制。




# 9. WeightNorm

## 9.1 数学

WeightNorm 将权重向量重参数化为

$$
w = g \frac{v}{\|v\|}
$$

其中：

- $v$ 控制方向
- $g$ 控制幅度

这将尺度和方向分离。

---

## 9.2 数值示例

设

$$
v = [3,4], \qquad \|v\|_2 = 5, \qquad g = 10
$$

则

$$
w = 10 \cdot [3/5,4/5] = [6,8]
$$

In [9]:
v = np.array([3., 4.])
g = 10.
w = g * v / np.linalg.norm(v)

print("WeightNorm example")
print("v =", v)
print("||v|| =", np.linalg.norm(v))
print("g =", g)
print("w =", w)

WeightNorm example
v = [3. 4.]
||v|| = 5.0
g = 10.0
w = [6. 8.]


## 9.3 WeightNorm 的放置位置

WeightNorm **不是**一个独立的激活归一化层。

它不是放在：

- 层之前
- 层之后

而是：

> **在层的参数化内部**

因此正确的理解方式是：

```text
Linear/Conv with reparameterized weights
```

---

## 9.4 WeightNorm 解决的问题

- 降低权重方向与幅度之间的耦合
- 可以改善优化行为
- 在某些场景下可以加速学习

---

## 9.5 使用 WeightNorm 的典型网络

历史上：

- RNN
- 一些 MLP
- 一些生成模型

如今它的地位不如 BN/LN/RMSNorm 重要。

---

## 9.6 没有 WeightNorm 时的情况

通常网络仍然可以训练，但优化可能会：

- 更慢
- 对学习率更敏感
- 在某些场景下几何性质较差

# 10. SpectralNorm (SN)

## 10.1 数学

谱归一化将权重矩阵除以其最大奇异值：

$$
\bar{W} = \frac{W}{\sigma_{\max}(W)}
$$

这约束了该层的算子范数。

---

## 10.2 数值示例

设

$$
W =
\begin{bmatrix}
3 & 0 \\
0 & 4
\end{bmatrix}
$$

奇异值为 $3$ 和 $4$，因此

$$
\sigma_{\max}(W)=4
$$

从而

$$
\bar{W} =
\begin{bmatrix}
0.75 & 0 \\
0 & 1
\end{bmatrix}
$$

现在最大放大倍数至多为 $1$。

In [10]:
W = np.array([[3., 0.], [0., 4.]])
smax = np.linalg.norm(W, 2)
Wbar = W / smax

print("SpectralNorm example")
print("W =")
print(W)
print("sigma_max =", smax)
print("W_bar =")
print(Wbar)
print("spectral norm of W_bar =", np.linalg.norm(Wbar, 2))

SpectralNorm example
W =
[[3. 0.]
 [0. 4.]]
sigma_max = 4.0
W_bar =
[[0.75 0.  ]
 [0.   1.  ]]
spectral norm of W_bar = 1.0


## 10.3 SpectralNorm 的放置位置

与 WeightNorm 类似，SpectralNorm 也不是一个独立的激活归一化层。

它直接作用于层的权重。

因此正确的理解是：

> **权重级别的约束/归一化**

而非「之前」或「之后」。

---

## 10.4 SpectralNorm 解决的问题

- 控制一层的最大放大倍数
- 改善 Lipschitz 性质
- 稳定对抗性或生成式训练
- 降低对扰动的极端敏感性

---

## 10.5 使用 SpectralNorm 的典型网络

- GAN 判别器
- 面向鲁棒性的模型
- 需要控制 Jacobian 放大的场景

---

## 10.6 没有 SpectralNorm 时的典型病症

常见问题：

- GAN 训练不稳定
- 判别器变得过于尖锐
- 生成器梯度饥饿（gradient starvation）
- 对小扰动过度敏感

---

## 10.7 SpectralNorm vs Per-feature 预处理 / BatchNorm：混合单位输入下用法

### 问题场景

在 vehicle control MPC 中，状态向量通常包含量纲和尺度差异极大的物理量：

$$
x = [\underbrace{e_y}_{\text{lateral error (m)} \sim 0.5}, \quad \underbrace{e_\psi}_{\text{heading error (rad)} \sim 0.05}, \quad \underbrace{v}_{\text{velocity (m/s)} \sim 30}, \quad \underbrace{\dot\psi}_{\text{yaw rate (rad/s)} \sim 0.1}]
$$

此时 $\|x\|_2 \approx 30.0$，几乎完全由 velocity 维度主导。

### SpectralNorm 为什么解决不了这个问题

SpectralNorm 将权重矩阵归一化为 $\bar{W} = W / \sigma_{\max}(W)$，使得 $\|\bar{W}\|_2 = 1$。但：

$$
y = \bar{W} x
$$

$\bar{W}$ 的谱范数是 1，但 $x$ 各维度的尺度差了两个数量级。输出 $y$ 仍然被 velocity 维度主导。

更关键的是，如果 $W$ 在 yaw rate 方向学到了一个大权重来补偿它的小尺度，SpectralNorm 会把这个有意义的大权重一起压下去——它**不区分**"因为 feature 尺度小所以权重大"和"权重真的太大了"。

| | SpectralNorm | Per-feature 预处理 / BatchNorm |
|---|---|---|
| **作用对象** | 权重矩阵 $W$ | 输入/激活 $x$ |
| **控制什么** | $W$ 的最大放大倍数 $\sigma_{\max}(W) = 1$ | 每个 feature channel 的均值和方差 |
| **能否解决 feature 尺度不均匀** | 不能 | 能 |

### 正确做法：per-feature standardization

用训练集统计量将每个 feature 归一化到 $\mathcal{O}(1)$：

$$
\hat{x}_c = \frac{x_c - \mu_c}{\sigma_c + \epsilon}
$$

归一化后：

$$
\hat{x} = [\underbrace{\hat{e}_y}_{\sim \mathcal{O}(1)}, \quad \underbrace{\hat{e}_\psi}_{\sim \mathcal{O}(1)}, \quad \underbrace{\hat{v}}_{\sim \mathcal{O}(1)}, \quad \underbrace{\hat{\dot\psi}}_{\sim \mathcal{O}(1)}]
$$

现在所有维度对 $Wx$ 的贡献在同一量级，Jacobian 的谱不再被单一维度主导。如果下游需要原始物理量纲（比如输出是 steering angle 或加速度命令），在最后一层 scale back 即可。

### 预处理 vs BatchNorm 的选择

| | 预处理 standardization | BatchNorm |
|---|---|---|
| **输入层** | 首选，简单、确定性、无 batch 依赖 | 也行，但多此一举 |
| **隐藏层** | 不适用 | 适用 |
| **推理一致性** | 完全一致 | 有 train/eval gap |
| **小 batch / batch=1** | 不受影响 | 统计量噪声大或退化 |

对于 MPC 状态输入，**预处理 standardization 是最直接可靠的方案**。

### 组合使用

两类方法不矛盾，各管各的层面：

```text
原始状态 [e_y, e_ψ, v, ψ̇]
    ↓
per-feature standardization   ← 解决 feature 尺度不均匀
    ↓
Linear (可选 SpectralNorm)    ← SpectralNorm 控制层增益
    ↓
...
```

- **预处理 / BatchNorm**：让输入各维度在同一尺度 → Jacobian 谱均匀、条件数好
- **SpectralNorm**：让每层不额外放大 → 深层稳定、Lipschitz 受控

两者解决的问题正交。但优先级是**先把输入搞平**，否则 SpectralNorm 管住了权重也无济于事。


# 11. 总结表：放置位置、解决的问题、适用结构、缺失时的后果

| 方法 | 数学思想 | 典型放置位置 | 主要解决的问题 | 常见结构 | 缺失时 |
|---|---|---|---|---|---|
| $L_1$ / $L_2$ / $L_\infty$ | 向量幅度 | 非层放置问题 | 正则化/约束/度量 | 所有 | 对尺度或稀疏性控制较弱 |
| Frobenius norm | 矩阵总能量 | 非层放置问题 | 权重幅度/误差能量 | 所有 | 尺度监控不良 |
| Spectral norm（数学） | 最大奇异值 | 非层放置问题 | 最大放大控制 | 所有 | 隐藏的放大可能被忽视 |
| BatchNorm | batch 均值/方差归一化 | **层之后、激活之前** | CNN 训练稳定性 | CNN, ResNet | 激活不稳定，优化更困难 |
| LayerNorm（MLP 形式） | 特征均值/方差归一化 | **层之后、激活之前** | 特征尺度稳定 | MLP，部分块 | 尺度漂移 |
| LayerNorm（Transformer） | 特征均值/方差归一化 | 通常在 Pre-Norm 中**子层之前** | 残差流稳定性 | Transformer, ViT | 深层扩展更困难 |
| RMSNorm | 基于 RMS 的缩放 | **子层之前** | 稳定深层残差 Transformer | LLMs | 残差幅度漂移 |
| GroupNorm | 分组归一化 | **层之后、激活之前** | 小 batch CNN 稳定性 | 检测/分割 CNN | 噪声较大的 BN 统计量 |
| InstanceNorm | 逐实例逐通道归一化 | **层之后、激活之前** | 风格统计量控制 | 风格迁移/生成 | 风格去除效果弱 |
| WeightNorm | 权重重参数化 | **在层权重内部** | 解耦幅度与方向 | 部分 RNN/MLP/生成模型 | 优化可能更慢 |
| SpectralNorm（层） | 除以最大奇异值 | **在层权重内部** | Lipschitz / 最大增益控制 | GAN / 鲁棒性 | 过于尖锐的映射 |


# 12. 不使用 Normalization 时的典型训练病症

## 12.1 激活爆炸

假设每层将信号范数大约放大 $3$ 倍。

经过 $10$ 层后：

$$
3^{10} \approx 59049
$$

激活值可以变得非常大。

后果：

- 数值不稳定
- 非线性饱和
- 梯度爆炸

---

## 12.2 激活消失

假设每层将信号缩小为 $0.5$ 倍。

经过 $10$ 层后：

$$
0.5^{10} \approx 0.00098
$$

表示几乎消失。

后果：

- 信号传播微弱
- 特征学习能力差
- 梯度消失

---

## 12.3 非线性饱和

对于 sigmoid：

$$
\sigma(z)=\frac{1}{1+e^{-z}}, \qquad \sigma'(z)=\sigma(z)(1-\sigma(z))
$$

当 $z=0$ 时：

$$
\sigma'(0)=0.25
$$

当 $z=10$ 时：

$$
\sigma(10)\approx 0.99995,\qquad \sigma'(10)\approx 0.00005
$$

因此如果激活前的值偏移过大，梯度就会崩塌。

---

## 12.4 不同网络的典型失效模式

### CNN 没有 BN/GN 时
- 激活漂移
- 梯度不稳定
- loss 震荡
- 超参数脆弱

### Transformer 没有 LN/RMSNorm 时
- 残差流尺度漂移
- attention logits 不稳定
- softmax 饱和
- 深层扩展不稳定

### GAN 没有 SpectralNorm 时
- 判别器过于尖锐
- 训练震荡
- 模式崩塌 / 生成器梯度饥饿


In [8]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [0, 10]:
    s = sigmoid(z)
    sp = s * (1 - s)
    print(f"z={z:>2}, sigmoid(z)={s:.8f}, derivative={sp:.8f}")

z= 0, sigmoid(z)=0.50000000, derivative=0.25000000
z=10, sigmoid(z)=0.99995460, derivative=0.00004540


# 13. 实用调试对照表

| 症状 | 可能的原因 | 排查方向 |
|---|---|---|
| loss 早期爆炸 | 激活/梯度尺度失控 | 是否缺少或错放了 BN/LN，初始化是否合理，LR 是否过大 |
| 很深的 CNN 训练效果差 | 激活前统计量漂移 | BN 放置位置，小 batch 时考虑 GN |
| 小 batch 视觉训练不稳定 | BN 统计量噪声过大 | 用 GN/LN 替换 BN |
| Transformer 随深度增加变得不稳定 | 残差流漂移 | 使用 Pre-LN 或 RMSNorm |
| attention logits 变得过于尖锐 | 隐状态尺度过大 | LN/RMSNorm 放置位置，残差缩放 |
| GAN 剧烈震荡 | 层增益失控 | SpectralNorm 或其他 Lipschitz 控制 |
| 训练缓慢但没有爆炸 | 参数几何性质差 | 考虑 WeightNorm / 初始化 / 优化器调优 |


# 14. 一般规则

## CNN 规则
通常：

```text
Conv / Linear → Norm → Activation
```

典型选择：

- 标准 CNN 用 BN
- 小 batch CNN 用 GN
- 风格敏感的图像生成用 IN

---

## Transformer / LLM 规则
通常：

```text
Norm → Attention / MLP → Residual Add
```

典型选择：

- Transformer 用 LayerNorm
- 许多现代 LLM 用 RMSNorm

---

## 权重级别方法
不存在「之前还是之后」的问题：

- WeightNorm
- SpectralNorm

它们直接修改或约束层的权重。


# 15. 最终总结

一个清晰的记忆方式：

- **数学范数**度量大小
- **归一化层**在训练中控制尺度/统计量
- **CNN 风格的归一化**通常放在线性/卷积层**之后**、激活函数**之前**
- **Transformer/LLM 的归一化**通常放在功能子层**之前**
- **WeightNorm 和 SpectralNorm** 作用于权重本身，而非作为独立的激活归一化模块

核心目标始终相同：

> 控制信号尺度、梯度尺度和优化几何

失去这种控制后，常见的病症包括：

- 激活爆炸
- 激活消失
- 梯度爆炸/消失
- 饱和
- 不稳定的残差流
- 脆弱或失败的训练
